In [1]:
!pip install -q openai requests


## 2. Config — add your API keys here

- `OPENAI_API_KEY` — from https://platform.openai.com/api-keys
- `GOOGLE_PLACES_API_KEY` — enable "Places API" in Google Cloud Console
- `SERPAPI_KEY` — from https://serpapi.com/ (used for general web + YouTube search)


In [ ]:
import os

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
GOOGLE_PLACES_API_KEY = os.getenv("GOOGLE_PLACES_API_KEY", "")
SERPAPI_KEY = os.getenv("SERPAPI_KEY", "")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY


In [ ]:
from openai import OpenAI

client = OpenAI(api_key=OPENAI_API_KEY)

def ask_llm(prompt, max_tokens=1000):
    response = client.chat.completions.create(
        model="gpt-4o",
        max_tokens=max_tokens,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content


In [ ]:
import json as pyjson

def get_required_documents(business_type, city):
    prompt = f"""You are a business licensing expert in India.

Business type: {business_type}
City/Location: {city}

List every document, license, or registration required to legally start and operate this business in this location.

Respond ONLY with valid JSON, no preamble, no markdown formatting, in this exact structure:
{{
  "documents": [
    {{
      "name": "Document/License name",
      "issuing_authority": "Who issues it",
      "why_needed": "One line reason",
      "typical_cost": "Approx cost or Free",
      "official_website": "Official portal URL if known, else empty string"
    }}
  ]
}}
"""
    raw = ask_llm(prompt, max_tokens=1500)
    cleaned = raw.replace("```json", "").replace("```", "").strip()
    try:
        return pyjson.loads(cleaned)
    except pyjson.JSONDecodeError:
        print("Could not parse JSON, raw output below:")
        print(raw)
        return {"documents": []}


In [ ]:
import requests

def search_places(query, city, max_results=5):
    url = "https://maps.googleapis.com/maps/api/place/textsearch/json"
    params = {
        "query": f"{query} consultant near {city}",
        "key": GOOGLE_PLACES_API_KEY
    }
    results = []
    try:
        resp = requests.get(url, params=params).json()
        for place in resp.get("results", [])[:max_results]:
            results.append({
                "name": place.get("name"),
                "address": place.get("formatted_address"),
                "rating": place.get("rating"),
                "user_ratings_total": place.get("user_ratings_total"),
            })
    except Exception:
        pass
    # If Google Places is not enabled on this key, use SerpAPI Google Maps engine
    if not results and SERPAPI_KEY:
        try:
            resp = requests.get("https://serpapi.com/search", params={
                "engine": "google_maps",
                "q": f"{query} consultant near {city}",
                "api_key": SERPAPI_KEY
            }).json()
            for place in resp.get("local_results", [])[:max_results]:
                results.append({
                    "name": place.get("title"),
                    "address": place.get("address"),
                    "rating": place.get("rating"),
                    "user_ratings_total": place.get("reviews"),
                })
        except Exception:
            pass
    return results


In [ ]:
def search_web(query, engine="google"):
    url = "https://serpapi.com/search"
    params = {
        "q": query,
        "engine": engine,
        "api_key": SERPAPI_KEY,
        "num": 5
    }
    resp = requests.get(url, params=params).json()
    results = []
    for item in resp.get("organic_results", [])[:5]:
        results.append({
            "title": item.get("title"),
            "link": item.get("link"),
            "snippet": item.get("snippet")
        })
    return results

def search_youtube(query):
    return search_web(f"{query} site:youtube.com", engine="google")


In [ ]:
def build_report(business_type, city):
    print("Step 1: Getting required documents...")
    docs_data = get_required_documents(business_type, city)
    documents = docs_data.get("documents", [])

    enriched = []
    for doc in documents:
        doc_name = doc.get("name", "")
        print(f"  -> Researching: {doc_name}")

        places = search_places(doc_name, city)
        web_results = search_web(f"how to apply for {doc_name} in {city} free")
        yt_results = search_youtube(f"how to apply for {doc_name} in {city}")

        enriched.append({
            **doc,
            "nearby_agents": places,
            "web_guides": web_results,
            "youtube_guides": yt_results
        })

    print("Step 4: Asking the LLM to format final report...")
    format_prompt = f"""You are formatting a business licensing report for a user opening a
"{business_type}" in {city}.

Here is the raw data (documents, nearby agents, web guides, YouTube guides) as JSON:

{pyjson.dumps(enriched, indent=2)}

Write a clean, easy-to-read report with:
- One section per document/license
- For each: what it is, cost, and the CHEAPEST/best way to get it
  (prefer free official routes over paid agents when possible)
- List 1-2 best nearby agents (name, rating, address) as a paid fallback option
- List 1-2 useful web/YouTube links
- Keep it practical and skimmable, use headers and bullet points
"""
    final_report = ask_llm(format_prompt, max_tokens=2000)
    return final_report, enriched


## 8. Run it

In [ ]:
business_type = "food stall"
city = "Mumbai"

report, raw_data = build_report(business_type, city)
print(report)
